# **Explore Gridded Datasets**

*Usually raster*

- Load GRID3 Gridded PEs
- Meta Data For Good High-Res Population Density (HRPD) data
- GHSL Datasets: POP (Population); Then BUILD-S (Built-up surface) and SMOD (Degree of Urbanization)
- Kontur: Population density hexagons (use h3?)


# **Setup**


In [ ]:
!pip install rasterio folium matplotlib mapclassify

In [ ]:
import rasterio
import pandas as pd
import geopandas as gpd
import plotly.express as px

In [ ]:
# Sample filepath / load: pd.read_csv("drive/MyDrive/data.csv")
from google.colab import drive
drive.mount('/content/drive')
data_dir = "drive/MyDrive/Colab Notebooks/Data/dd-afkenya/"
data_dir

# **Reference**

In [ ]:
# TBD

# **Config**

In [ ]:
sample_counties = [
    "Uasin Gishu"
]

In [ ]:
# Projected CRS (enabling lat/lon distance comparisons in human units eg: KM)
PROJ_CRS = "EPSG:21037"

# Non-projected general CRS
GNRL_CRS = "EPSG:4326"

# **Load Data**

## **Admin Boundaries (COD)**

In [ ]:
adm_file = data_dir + "ken_adm_iebc_20191031_shp.zip"
adm_df = gpd.read_file(adm_file, layer="ken_admbnda_adm1_iebc_20191031").to_crs(crs=PROJ_CRS)
sample_adm_df = adm_df[adm_df["ADM1_EN"].isin(sample_counties)].copy()

adm_df.shape, sample_adm_df.shape

In [ ]:
# sample_adm_df.explore(style_kwds=dict(color="red", opacity=0.1, fillOpacity=0.1), tiles="CartoDB positron", highlight_kwds=dict(fillOpacity=0.1))

## **Populated Places (HOTOSM)**

In [ ]:
# HOTOSM Populated Places
hotosm_file = data_dir + "hotosm_ken_populated_places_points_shp.zip"
hotosm_df = gpd.read_file(hotosm_file).to_crs(crs=PROJ_CRS)

# Drop isolated dwellings
# hotosm_df = hotosm_df[~hotosm_df["place"].isin(["isolated_dwelling"])]

# Sample by intersection w/Adm. boundaries
sample_hotosm_df = hotosm_df[hotosm_df.apply(lambda r: r["geometry"].intersects(sample_adm_df.geometry).sum() > 0, axis=1)].copy()

hotosm_df.shape, sample_hotosm_df.shape

In [ ]:
# sample_hotosm_df.explore(color="darkorchid", marker_kwds=dict(radius=5))

## **Aquaya Waterpoints**

Systems and Labs

In [ ]:
# Waterpoints
wp_file = data_dir + "AF Kenya - Consolidated Water Systems.xlsx"
wp_df = pd.read_excel(wp_file, sheet_name="Systems")
wp_df = gpd.GeoDataFrame(wp_df, geometry=gpd.points_from_xy(wp_df["Longitude"], wp_df["Latitude"], crs=GNRL_CRS)).to_crs(crs=PROJ_CRS)
sample_wp_df = wp_df[wp_df["County"].isin(sample_counties)].copy()

wp_df.shape, sample_wp_df.shape

In [ ]:
# Labs
labs_df = pd.read_excel(wp_file, sheet_name="Labs")
labs_df = gpd.GeoDataFrame(labs_df, geometry=gpd.points_from_xy(labs_df["Longitude"], labs_df["Latitude"], crs="EPSG:4326")).to_crs(crs=PROJ_CRS)
sample_labs_df = labs_df[labs_df["County"].isin(sample_counties)].copy()

labs_df.shape, sample_labs_df.shape

In [ ]:
# wp_m = wp_df.explore(color="deepskyblue", marker_kwds=dict(radius=10), tiles="CartoDB positron")
# wp_m = labs_df.explore(m=wp_m, color="tomato", marker_kwds=dict(radius=10))
# wp_m

## **GRID3 Pop Grids**

In [ ]:
grid3_grid_file = data_dir + "grid3_ken_settlement_grid_v3_0/GRID3_KEN_settlement_grid_v3_0.gpkg"

# Takes a LONG time - ~15 mins. 5,070,879 rows.
grid3_grid_df = gpd.read_file(grid3_grid_file).to_crs(crs=PROJ_CRS)

# Sample down to county - use a spatial join for efficiency with large datasets. Sample for Uasin Gishu is 152,150 rows
sample_grid3_grid_df = gpd.sjoin(grid3_grid_df, sample_adm_df, how="inner", predicate="intersects")

grid3_grid_df.shape, sample_grid3_grid_df.shape

In [ ]:
sample_grid3_grid_df.head(2)

In [ ]:
# Can we plot it? Or death (too much?)
# m = sample_grid3_grid_df.explore()
# m

## **PopCluster - Kenya**

Publication's population clusters (custom alg. derived specifically for Sub-Saharan Africa c. 2020 with all kinds of datasets - see notes and publication).

Population clusters for sub-Saharan Africa. The population clusters include the following data:
```
1.	id – The IDs are given as a unique number for each cluster
2.  Country - Name of the country.
3.	Population – This is the population in each cluster obtained from the population dataset (GHSL for Somalia, Sudan and South Sudan and HRSL for the rest of the countries). The population in these clusters are calibrated to 2016 population values
4.	NightLight – This value is obtained from the night-time light map and represents the maximum luminance detected in each cluster based on the 2016 stable light product available.
5.	ElecPop – The number of people in each cluster who live in areas in which the stable light product detect light sources.
6.	Area – The area of each cluster given in square kilometrers.
7.	IsUrban – Classifies areas as either urban (2), peri-urban (1) or rural (0).
```



In [ ]:
popclusters_file = data_dir + "PopClusters - SSA - Kenya"
popclusters_df = gpd.read_file(popclusters_file).to_crs(crs=PROJ_CRS)

popclusters_df.shape

In [ ]:
# Sample down to county - use a spatial join for efficiency with large datasets. Sample for Uasin Gishu is 152,150 rows
sample_popclusters_df = gpd.sjoin(popclusters_df, sample_adm_df, how="inner", predicate="intersects")
sample_popclusters_df.shape

In [ ]:
sample_popclusters_df.head(2)

In [ ]:
sample_popclusters_df["IsUrban"].value_counts()

In [ ]:
fig = px.histogram(sample_popclusters_df, x="Population", facet_col="IsUrban")
fig.update_xaxes(matches=None)
fig.update_yaxes(matches=None)
fig.show()

In [ ]:
# Create a temporary column for coloring
sample_popclusters_df["color_id"] = sample_popclusters_df['id'] % 20

# Now, explore using the transformed ID for coloring
m = sample_popclusters_df.explore(column="color_id", cmap="tab20")
m

In [ ]:
pop_min, pop_max = 200, 5000
sample_popclusters_popfilt_df = sample_popclusters_df[sample_popclusters_df["Population"].between(pop_min, pop_max)]
sample_popclusters_popfilt_df.explore(column="color_id", cmap="tab20")

In [ ]:
sample_popclusters_popfilt_df.shape

# **Group "Sequential" GRID3 Grid Points**

Test this grouping method to see about "community" groups it creates...

- Group all points together that are within 105m of each other (includes square connections, excludes diagonals. Potential points are ~100m apart; extra 5m gives them some buffer)

In [ ]:
from shapely.geometry import Polygon
from shapely.ops import unary_union

# Define the distance threshold for grouping (in meters)
threshold = 105

# Buffer each point by the threshold distance
buffered_points = sample_grid3_grid_df.buffer(threshold)

# Use unary_union to merge overlapping buffers
merged_buffers = unary_union(buffered_points)

# If the merged buffers result in a MultiPolygon, convert it to a list of polygons
if merged_buffers.geom_type == 'MultiPolygon':
    grouped_polygons = list(merged_buffers.geoms)
else:
    grouped_polygons = [merged_buffers]

# Create a new GeoDataFrame from the grouped polygons
grid3_grouped_df = gpd.GeoDataFrame(geometry=grouped_polygons, crs=PROJ_CRS)

# Add a unique ID for each grouped polygon
grid3_grouped_df['group_id'] = grid3_grouped_df.index

grid3_grouped_df.shape

In [ ]:
grid3_grouped_df.explore(column="group_id", cmap="tab20")